# ML Predictors for Moran Process Fixation Outcomes

Trains Linear Regression and XGBoost models to predict fixation probability or fixation time from graph structural properties.

**To run:** set `BATCH_NAME` and `TARGET_COLUMN` in the Setup cell below, then run all cells top to bottom. Re-run with the other `TARGET_COLUMN` to train both models. Saved models go to `{BATCH_DIR}/ml_models/` and can be loaded by `experiment_analysis.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys
sys.path.insert(0, 'src')
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib

from moran_process.analysis.analysis_utils import GRAPH_PROPERTY_COLUMNS

# ── Configuration -- only edit these ──────────────────────────────────────────
BATCH_NAME    = 'batch_large_test_30_02'  # must match experiment_analysis.ipynb
TARGET_COLUMN = 'prob_fixation'                # 'prob_fixation' or 'mean_steps'
R_FILTER      = 1.1   # None = train across all r values (r becomes a feature)
                       # e.g. 1.1 = train on one r slice only
# ──────────────────────────────────────────────────────────────────────────────

ROOT      = Path(os.getcwd())
BATCH_DIR = ROOT / "simulation_data" / BATCH_NAME
DATA_PATH = BATCH_DIR / "graph_statistics.csv"

print(f"Batch : {BATCH_NAME}")
print(f"Target: {TARGET_COLUMN}")
print(f"r     : {'all' if R_FILTER is None else R_FILTER}")

In [ ]:
# Drop graph-type-specific metadata and columns too sparse for general prediction
drop_raw = ['graph6_string', 'branching', 'depth', 'n_rods', 'rods_length', 'rod_length', 'seed', 'n_grouped']

df = pd.read_csv(DATA_PATH).drop(columns=drop_raw, errors='ignore')
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
if R_FILTER is not None:
    df_model     = df[df['r'] == R_FILTER].copy()
    feature_cols = [c for c in GRAPH_PROPERTY_COLUMNS if c in df.columns]
else:
    df_model     = df.copy()
    feature_cols = ['r'] + [c for c in GRAPH_PROPERTY_COLUMNS if c in df.columns]

X = df_model[feature_cols].select_dtypes(include=[np.number])
if 'density' in X.columns:
    X = X.drop(columns=['density'])
y = df_model[TARGET_COLUMN]

X = X.fillna(X.median())

print(f"Shape  : {X.shape[0]} samples x {X.shape[1]} features")
print(f"NaNs   : {X.isna().sum().sum()} total across all features")
print(f"Target : {y.min():.4f} -- {y.max():.4f}  (mean {y.mean():.4f})")
print("Features:", list(X.columns))

## Linear Regression Baseline

Standardizes features then fits a linear model. Standardized coefficients give a direct interpretability signal -- the largest absolute values are the graph properties that most linearly drive the target. Compare these with the XGBoost SHAP values below: agreement between the two is stronger evidence that a property genuinely matters.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

std_lr = make_pipeline(StandardScaler(), LinearRegression())
std_lr.fit(X_train, y_train)
lr_preds = std_lr.predict(X_test)

coefficients = std_lr.named_steps['linearregression'].coef_
std_coef_df = pd.DataFrame({'feature': X.columns, 'std_coefficient': coefficients})

model_filename = BATCH_DIR / "ml_models" / f'{TARGET_COLUMN}_linear_regression_pipeline.joblib'
os.makedirs(os.path.dirname(model_filename), exist_ok=True)
joblib.dump(std_lr, model_filename)
print(f"Saved: {model_filename}")

In [ ]:

print(f"LR R^2: {r2_score(y_test, lr_preds):.4f}")
print(f"LR RMSE: {root_mean_squared_error(y_test, lr_preds):.2f}")
 
# 1. Get absolute values of standardized coefficients
std_coef_df['abs_coefficient'] = std_coef_df['std_coefficient'].abs()

# 2. Calculate percentage
total_magnitude = std_coef_df['abs_coefficient'].sum()
std_coef_df['contribution_pct'] = (std_coef_df['abs_coefficient'] / total_magnitude) * 100

# 3. Sort and show
presentation_df = std_coef_df.sort_values(by='contribution_pct', ascending=False)
presentation_df['feature'] = presentation_df['feature'].str.replace('_', ' ').str.title()
print(presentation_df[['feature',  'contribution_pct', 'std_coefficient',]].head(10).to_string(float_format=lambda x: f'{x:.2f}'))

In [ ]:
plt.figure(figsize=(8, 6))
plt.hexbin(y_test, lr_preds, gridsize=30, cmap='Blues', mincnt=1)
plt.colorbar(label='Count')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel(f'Actual {TARGET_COLUMN}')
plt.ylabel(f'Predicted {TARGET_COLUMN}')
plt.title(f'Linear Regression: Predicted vs Actual ({TARGET_COLUMN})')
plt.legend()
plt.grid(True, alpha=0.3)
corr, p_value = pearsonr(lr_preds, y_test)
plt.text(0.05, 0.95, f'Pearson r = {corr:.4f}\np = {p_value:.2e}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

In [ ]:
scaler   = std_lr.named_steps['standardscaler']
lr_model = std_lr.named_steps['linearregression']

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

fast_explainer  = shap.LinearExplainer(lr_model, X_train_scaled)
fast_shap_values = fast_explainer(X_test_scaled)

shap.summary_plot(fast_shap_values, X_test_scaled)

In [ ]:
shap.plots.beeswarm(fast_shap_values, max_display=10)

## XGBoost Model

A gradient-boosted tree ensemble. Unlike LR, it captures non-linear interactions between graph properties without needing feature scaling. If XGBoost dramatically outperforms LR, that suggests the relationship between graph structure and fixation outcome is non-linear.

In [ ]:
mean_val  = y_train.mean()
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    objective='reg:squarederror',
    n_jobs=1,
    random_state=42,
    base_score=mean_val,
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_model_filename = BATCH_DIR / "ml_models" / f'{TARGET_COLUMN}_xgboost_model.joblib'
joblib.dump(xgb_model, xgb_model_filename)
print(f"Saved: {xgb_model_filename}")

In [ ]:
print(f"XGB R^2:  {r2_score(y_test, xgb_preds):.4f}")
print(f"XGB RMSE: {root_mean_squared_error(y_test, xgb_preds):.4f}")

plt.figure(figsize=(8, 6))
plt.hexbin(y_test, xgb_preds, gridsize=30, cmap='Reds', mincnt=1)
plt.colorbar(label='Count')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel(f'Actual {TARGET_COLUMN}')
plt.ylabel(f'Predicted {TARGET_COLUMN}')
plt.title(f'XGBoost: Predicted vs Actual ({TARGET_COLUMN})')
plt.legend()
plt.grid(True, alpha=0.3)
corr, p_value = pearsonr(y_test, xgb_preds)
plt.text(0.05, 0.95, f'Pearson r = {corr:.4f}\np = {p_value:.2e}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

## Model Agreement

If LR and XGBoost make similar predictions despite very different inductive biases, the signal is likely robust rather than an artifact of one model's assumptions.

In [ ]:
plt.figure(figsize=(8, 6))
plt.hexbin(lr_preds, xgb_preds, gridsize=30, cmap='Purples', mincnt=1)
plt.colorbar(label='Count')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Full agreement')
plt.xlabel(f'LR prediction ({TARGET_COLUMN})')
plt.ylabel(f'XGBoost prediction ({TARGET_COLUMN})')
plt.title(f'LR vs XGBoost: {TARGET_COLUMN}')
plt.legend()
plt.grid(True, alpha=0.3)
corr, p_value = pearsonr(lr_preds, xgb_preds)
plt.text(0.05, 0.95, f'Pearson r = {corr:.4f}\np = {p_value:.2e}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

In [ ]:
explainer   = shap.PermutationExplainer(xgb_model.predict, X_test)
shap_values = explainer(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, show=False)
plt.title(f'XGBoost Feature Importance ({TARGET_COLUMN})')
plt.show()

top_feature_idx  = np.abs(shap_values.values).mean(0).argmax()
top_feature_name = X_test.columns[top_feature_idx]
print(f"Top feature: {top_feature_name}")

plt.figure()
shap.dependence_plot(top_feature_name, shap_values.values, X_test, show=False)
plt.show()

## Cross-Model Summary

Loads all saved models and evaluates them on the full dataset. Run this section only after training both targets (`prob_fixation` and `mean_steps`) -- all four `.joblib` files must exist in `{BATCH_DIR}/ml_models/`.

In [ ]:
from sklearn.metrics import r2_score
import matplotlib.lines as mlines
import seaborn as sns
from moran_process.analysis.analysis_utils import COLOR_DICT, generate_robust_color_dict

ML_MODELS_DIR = BATCH_DIR / "ml_models"

MODEL_TYPE_MAP = {
    'linear_regression': 'LR',
    'xgboost':           'XGBoost',
}

all_models = {}
for path in sorted(ML_MODELS_DIR.glob("*.joblib")):
    stem           = path.stem  # e.g. 'mean_steps_linear_regression_pipeline'
    model_type_key = next((k for k in MODEL_TYPE_MAP if k in stem), None)
    if model_type_key is None:
        print(f"Skipping unrecognised model file: {path.name}")
        continue
    target     = stem[: stem.index(model_type_key) - 1]  # everything before the model type keyword
    model_type = MODEL_TYPE_MAP[model_type_key]
    all_models[(target, model_type)] = joblib.load(path)

expected_features = list(next(iter(all_models.values())).feature_names_in_)
print(f"Loaded {len(all_models)} models:")
for target, model_type in sorted(all_models):
    print(f"  {target:20} | {model_type}")

In [ ]:
X_eval = df[[c for c in expected_features if c in df.columns]].copy()
X_eval = X_eval.fillna(X_eval.median())

category_color_dict = generate_robust_color_dict(df, COLOR_DICT)

def plot_pred_vs_true(ax, true_vals, pred_vals, categories, n_nodes, title):
    plot_df = pd.DataFrame({
        'true':     true_vals.values,
        'pred':     pred_vals,
        'category': categories.values,
        'n_nodes':  n_nodes.values,
    })
    sns.scatterplot(
        data=plot_df, x='true', y='pred',
        hue='category', size='n_nodes',
        sizes=(30, 300), palette=category_color_dict,
        alpha=0.7, edgecolor='w', ax=ax, legend=False,
    )
    r2   = r2_score(true_vals, pred_vals)
    lims = [min(true_vals.min(), pred_vals.min()), max(true_vals.max(), pred_vals.max())]
    ax.plot(lims, lims, 'k--', alpha=0.5)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('True', fontsize=11)
    ax.set_ylabel('Predicted', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.text(0.05, 0.95, f'$R^2$ = {r2:.3f}', transform=ax.transAxes, fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9))

unique_targets     = sorted(set(t for t, _ in all_models))
unique_model_types = sorted(set(m for _, m in all_models))
n_rows, n_cols     = len(unique_targets), len(unique_model_types)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 6 * n_rows), squeeze=False)
fig.suptitle('Predicted vs True: All Models on Full Dataset', fontsize=16)

for row_idx, target in enumerate(unique_targets):
    for col_idx, model_type in enumerate(unique_model_types):
        ax = axes[row_idx, col_idx]
        if (target, model_type) not in all_models:
            ax.set_visible(False)
            continue
        title = f"{model_type}: {target.replace('_', ' ').title()}"
        preds = all_models[(target, model_type)].predict(X_eval)
        plot_pred_vs_true(ax, df[target], preds, df['category'], df['n_nodes'], title)

present_cats = sorted(df['category'].dropna().unique())
handles = [
    mlines.Line2D([], [], marker='o', color='w',
                  markerfacecolor=category_color_dict.get(c, 'gray'), markersize=10, label=c)
    for c in present_cats
]
handles.append(mlines.Line2D([], [], color='k', linestyle='--', alpha=0.5, label='Ideal (y=x)'))
fig.legend(handles=handles, title='Category', loc='center left',
           bbox_to_anchor=(0.87, 0.5), fontsize=11, title_fontsize=12)

plt.subplots_adjust(left=0.07, right=0.86, top=0.92, bottom=0.07, wspace=0.2, hspace=0.3)
plt.show()